# Evaluation Rigor Smoke Test

Loads one real AIST++ SMPL sequence from `config.yaml`, creates a degraded noisy copy, and verifies that TSI/JLVR/BAS plus failure-case mining expose the quality drop.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from embodied_motion_flow.config import load_config
from embodied_motion_flow.data.aist_loader import build_aist_dataset
from embodied_motion_flow.diagnostics.failure_analysis import FailureCaseAnalyzer
from embodied_motion_flow.evaluation.metrics import (
    beat_alignment_score,
    default_smpl_joint_limits,
    joint_limit_violation_rate,
    temporal_smoothness_index,
)
from embodied_motion_flow.reproducibility import set_global_seed
from embodied_motion_flow.utils.device import resolve_device

cfg = load_config("config.yaml")
set_global_seed(cfg.reproducibility.seed, cfg.reproducibility.deterministic_torch, cfg.reproducibility.benchmark_cudnn)
device = resolve_device(cfg.device.preference)
print(f"device={device}")

In [ ]:
dataset = build_aist_dataset(cfg.data.aist, cfg.data.sequence_length, split="val")
sample = dataset[0]
clean = sample["motion"].unsqueeze(0).to(device)
source_path = sample["source_path"]
print(f"Loaded {source_path}")
print(f"clean shape={tuple(clean.shape)}")

In [ ]:
torch.manual_seed(123)
noisy = clean.clone() + 0.04 * torch.randn_like(clean)

# Inject an off-beat extremity spike and anatomical limit violations.
offbeat_frame = min(47, clean.shape[1] - 2)
right_wrist_axis = 21 * 3
left_knee_axis = 4 * 3
noisy[:, offbeat_frame : offbeat_frame + 2, right_wrist_axis : right_wrist_axis + 3] += 3.5
noisy[:, :, left_knee_axis] += 3.0

beat_frames = torch.arange(15, clean.shape[1], 30, device=device).view(1, -1)
lower, upper = default_smpl_joint_limits(device=device)

rows = []
for name, motion in [("real_aist", clean), ("noisy_degraded", noisy)]:
    rows.append(
        {
            "sequence": name,
            "TSI": float(temporal_smoothness_index(motion, reduce="mean").item()),
            "JLVR": float(joint_limit_violation_rate(motion, lower, upper, reduce="mean").item()),
            "BAS": float(beat_alignment_score(motion, beat_frames, tolerance_frames=3, reduce="mean").item()),
        }
    )

df = pd.DataFrame(rows)
df

In [ ]:
ax = df.set_index("sequence")[["TSI", "JLVR", "BAS"]].plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title("Quantitative degradation diagnostics")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
failure_dir = Path(cfg.project.output_dir) / "failure_cases" / "notebook_rigor"
analyzer = FailureCaseAnalyzer(
    output_dir=failure_dir,
    tsi_threshold=cfg.evaluation.tsi_failure_threshold,
    jlvr_threshold=cfg.evaluation.jlvr_failure_threshold,
    self_collision_threshold=cfg.evaluation.self_collision_failure_threshold,
)
records = analyzer.analyze_batch(
    motion=noisy,
    tsi=temporal_smoothness_index(noisy, reduce="none"),
    jlvr=joint_limit_violation_rate(noisy, lower, upper, reduce="none"),
    self_collision=torch.zeros(noisy.shape[0], device=device),
    epoch=0,
    split="notebook",
    source_paths=[source_path],
)
print(f"Saved {len(records)} failure case(s) under {failure_dir}")
pd.read_csv(failure_dir / "failure_log.csv")